## Hidden Markov Models and Sequence Labeling
### Practical Python Activity
Students can create a simple sequence-labeling dataset.


In [1]:
training_data = [
    [("The", "DT"),
     ("student", "NN"),
     ("reads", "VB")],

    [("The", "DT"),
     ("teacher", "NN"),
     ("teaches", "VB")],

    [("A", "DT"),
     ("student", "NN"),
     ("writes", "VB")]
]

for sentence in training_data:
    words = [word for word, tag in sentence]
    tags = [tag for word, tag in sentence]

    print("Words:", words)
    print("Tags:", tags)

Words: ['The', 'student', 'reads']
Tags: ['DT', 'NN', 'VB']
Words: ['The', 'teacher', 'teaches']
Tags: ['DT', 'NN', 'VB']
Words: ['A', 'student', 'writes']
Tags: ['DT', 'NN', 'VB']


**Expected output:**
```
Words: ['The', 'student', 'reads']
Tags: ['DT', 'NN', 'VB']
Words: ['The', 'teacher', 'teaches']
Tags: ['DT', 'NN', 'VB']
Words: ['A', 'student', 'writes']
Tags: ['DT', 'NN', 'VB']
```

### From dataset to a trained tagger
The tiny dataset above is exactly the shape NLTK's `HiddenMarkovModelTrainer`
expects: a list of sentences, each a list of `(word, tag)` pairs. Training
estimates transition probabilities (tag → tag) and emission probabilities
(tag → word) from these counts, then Viterbi decoding picks the most likely
tag sequence for a new, unseen sentence.

In [2]:
from nltk.tag import hmm

trainer = hmm.HiddenMarkovModelTrainer()
tagger = trainer.train_supervised(training_data)

test_sentence = ["The", "teacher", "writes"]
print(tagger.tag(test_sentence))

[('The', 'DT'), ('teacher', 'NN'), ('writes', 'VB')]


/usr/local/lib/python3.12/dist-packages/nltk/tag/hmm.py:333: RuntimeWarning: overflow encountered in cast
  X[i, j] = self._transitions[si].logprob(self._states[j])
/usr/local/lib/python3.12/dist-packages/nltk/tag/hmm.py:335: RuntimeWarning: overflow encountered in cast
  O[i, k] = self._output_logprob(si, self._symbols[k])
/usr/local/lib/python3.12/dist-packages/nltk/tag/hmm.py:331: RuntimeWarning: overflow encountered in cast
  P[i] = self._priors.logprob(si)


Even though `"The teacher writes"` never appeared verbatim in training, the
model tags it correctly because it learned the DT → NN → VB transition
pattern and the word-tag emissions (`teacher`→NN, `writes`→VB) separately,
then combined them.

### Inspecting the learned probabilities (optional)
You can look inside the model to see exactly what it learned.

In [3]:
print("Tag set:", tagger._states)
print()
print("P(NN -> VB) transition prob:",
      tagger._transitions["NN"].prob("VB"))
print("P('writes' | VB) emission prob:",
      tagger._output_logprob("VB", "writes"))

Tag set: ['DT', 'NN', 'VB']

P(NN -> VB) transition prob: 1.0
P('writes' | VB) emission prob: -1.5849625007211563


---
## Week 3 Tool Activity (#19)
Builds tag transition counts and word/label emission counts directly from a small labelled dataset (no NLTK HMM classes), then uses those counts to tag a new sentence.

In [4]:
"""
Week 3 Tool Activity (#19)
A minimal HMM-style tagger built from raw counts (no NLTK HMM classes).

Steps:
1. Accept a labelled training dataset.
2. Extract words.
3. Extract labels.
4. Count transitions (tag -> tag).
5. Count word-label combinations (emissions).
6. Calculate basic probabilities.
7. Use the probabilities to analyse (tag) a new sentence.
"""

from collections import Counter, defaultdict

# 1. Labelled training dataset
# NOTE: the original 3 sentences from the lecture example are all exactly
# 3 tokens long (DT NN VB), so the model never sees what comes *after* a
# VB and cannot know that a noun like "books" should follow. One extra
# 4-token sentence is added so the model has real evidence for the
# VB -> NN transition and for the word "books" being a noun.
training_data = [
    [("The", "DT"), ("student", "NN"), ("reads", "VB")],
    [("The", "DT"), ("teacher", "NN"), ("teaches", "VB")],
    [("A", "DT"), ("student", "NN"), ("writes", "VB")],
    [("The", "DT"), ("teacher", "NN"), ("reads", "VB"), ("books", "NN")],
]

START = "<s>"


def extract_words_and_labels(data):
    all_words, all_labels = [], []
    for sentence in data:
        words = [w for w, t in sentence]
        labels = [t for w, t in sentence]
        all_words.append(words)
        all_labels.append(labels)
    return all_words, all_labels


def count_transitions(label_sequences):
    transition_counts = defaultdict(Counter)
    for labels in label_sequences:
        prev = START
        for tag in labels:
            transition_counts[prev][tag] += 1
            prev = tag
    return transition_counts


def count_emissions(data):
    emission_counts = defaultdict(Counter)
    for sentence in data:
        for word, tag in sentence:
            emission_counts[tag][word.lower()] += 1
    return emission_counts


def transition_prob(transition_counts, prev_tag, tag):
    counts = transition_counts[prev_tag]
    total = sum(counts.values())
    if total == 0:
        return 0.0
    return counts[tag] / total


def emission_prob(emission_counts, tag, word):
    counts = emission_counts[tag]
    total = sum(counts.values())
    if total == 0:
        return 0.0
    return counts[word.lower()] / total


def predict_tags(sentence_words, transition_counts, emission_counts, tagset):
    predicted = []
    prev_tag = START
    for word in sentence_words:
        scores = {}
        for tag in tagset:
            t_prob = transition_prob(transition_counts, prev_tag, tag)
            e_prob = emission_prob(emission_counts, tag, word)
            scores[tag] = t_prob * e_prob

        if all(score == 0 for score in scores.values()):
            # Backoff for an unseen word: emission is 0 for every tag,
            # so fall back to transition probability alone.
            scores = {tag: transition_prob(transition_counts, prev_tag, tag)
                      for tag in tagset}

        best_tag = max(scores, key=scores.get)
        predicted.append(best_tag)
        prev_tag = best_tag
    return predicted


if __name__ == "__main__":
    words, labels = extract_words_and_labels(training_data)
    print("Words per sentence:", words)
    print("Labels per sentence:", labels)

    transition_counts = count_transitions(labels)
    emission_counts = count_emissions(training_data)
    tagset = sorted({tag for seq in labels for tag in seq})

    print("\nTransition counts:")
    for prev, counts in transition_counts.items():
        print(f"  {prev}: {dict(counts)}")

    print("\nEmission counts:")
    for tag, counts in emission_counts.items():
        print(f"  {tag}: {dict(counts)}")

    print("\nSample probabilities:")
    print("  P(NN -> VB) =", round(transition_prob(transition_counts, "NN", "VB"), 3))
    print("  P('student' | NN) =", round(emission_prob(emission_counts, "NN", "student"), 3))

    test_sentence = ["The", "student", "reads", "books"]
    prediction = predict_tags(test_sentence, transition_counts, emission_counts, tagset)

    print("\nSentence:", " ".join(test_sentence))
    print("Predicted sequence:", " ".join(prediction))


Words per sentence: [['The', 'student', 'reads'], ['The', 'teacher', 'teaches'], ['A', 'student', 'writes'], ['The', 'teacher', 'reads', 'books']]
Labels per sentence: [['DT', 'NN', 'VB'], ['DT', 'NN', 'VB'], ['DT', 'NN', 'VB'], ['DT', 'NN', 'VB', 'NN']]

Transition counts:
  <s>: {'DT': 4}
  DT: {'NN': 4}
  NN: {'VB': 4}
  VB: {'NN': 1}

Emission counts:
  DT: {'the': 3, 'a': 1}
  NN: {'student': 2, 'teacher': 2, 'books': 1}
  VB: {'reads': 2, 'teaches': 1, 'writes': 1}

Sample probabilities:
  P(NN -> VB) = 1.0
  P('student' | NN) = 0.4

Sentence: The student reads books
Predicted sequence: DT NN VB NN


### How the system arrived at `DT NN VB NN`
- **The → DT, student → NN, reads → VB**: each of these words was seen in training with only one tag, so both the transition probability (`<s>→DT`, `DT→NN`, `NN→VB` are all 1.0 in this tiny dataset) and the emission probability agree, and the tagger has no ambiguity to resolve.
- **books → NN**: `books` never appeared in the first three lecture sentences, so one extra training sentence (*"The teacher reads books"*) was added to give the model real evidence that a VB can be followed by an NN and that `books` is a noun. Without that extra sentence, `VB` has no recorded transition at all (every original sentence ends right after the verb), so the tagger has no signal for what should come next and the choice collapses to an arbitrary tie-break — a good illustration of why HMM taggers need enough context in the training data, not just enough sentences.

---
## Week 3 Take-Home Project: HMM-Based Sequence Labeling System
A larger version of the same idea: 32 labelled training sentences, Laplace-smoothed transition/emission probabilities, full Viterbi decoding, and evaluation (accuracy + error analysis) on a held-out test set. This is also where the week 2 n-gram work and the week 3 HMM work meet: the same word-sequence patterns n-grams count directly are what the HMM's transition probabilities capture implicitly.

## 1–3. Training data, word/label extraction

In [5]:
from collections import Counter, defaultdict

START = "<s>"
END = "</s>"

# ---------------------------------------------------------------------------
# 1. Labelled training dataset (32 sentences, >= the required 30)
# ---------------------------------------------------------------------------
TRAINING_DATA = [
    [("the", "DT"), ("student", "NN"), ("reads", "VB"), ("books", "NN")],
    [("the", "DT"), ("teacher", "NN"), ("teaches", "VB"), ("students", "NN")],
    [("a", "DT"), ("student", "NN"), ("writes", "VB"), ("code", "NN")],
    [("the", "DT"), ("developer", "NN"), ("writes", "VB"), ("clean", "JJ"), ("code", "NN")],
    [("a", "DT"), ("developer", "NN"), ("builds", "VB"), ("a", "DT"), ("system", "NN")],
    [("the", "DT"), ("engineer", "NN"), ("designs", "VB"), ("a", "DT"), ("network", "NN")],
    [("the", "DT"), ("professor", "NN"), ("tests", "VB"), ("a", "DT"), ("program", "NN")],
    [("a", "DT"), ("student", "NN"), ("studies", "VB"), ("the", "DT"), ("algorithm", "NN")],
    [("the", "DT"), ("student", "NN"), ("runs", "VB"), ("the", "DT"), ("program", "NN")],
    [("the", "DT"), ("developer", "NN"), ("debugs", "VB"), ("the", "DT"), ("application", "NN")],
    [("a", "DT"), ("teacher", "NN"), ("creates", "VB"), ("a", "DT"), ("exam", "NN")],
    [("the", "DT"), ("engineer", "NN"), ("compiles", "VB"), ("the", "DT"), ("code", "NN")],
    [("the", "DT"), ("student", "NN"), ("submits", "VB"), ("the", "DT"), ("report", "NN")],
    [("a", "DT"), ("professor", "NN"), ("analyzes", "VB"), ("the", "DT"), ("dataset", "NN")],
    [("the", "DT"), ("student", "NN"), ("presents", "VB"), ("the", "DT"), ("project", "NN")],
    [("the", "DT"), ("teacher", "NN"), ("uses", "VB"), ("a", "DT"), ("laptop", "NN")],
    [("a", "DT"), ("developer", "NN"), ("uses", "VB"), ("a", "DT"), ("database", "NN")],
    [("the", "DT"), ("student", "NN"), ("writes", "VB"), ("a", "DT"), ("simple", "JJ"), ("program", "NN")],
    [("the", "DT"), ("engineer", "NN"), ("builds", "VB"), ("a", "DT"), ("new", "JJ"), ("server", "NN")],
    [("a", "DT"), ("student", "NN"), ("reads", "VB"), ("a", "DT"), ("difficult", "JJ"), ("book", "NN")],
    [("the", "DT"), ("developer", "NN"), ("designs", "VB"), ("a", "DT"), ("complex", "JJ"), ("system", "NN")],
    [("the", "DT"), ("professor", "NN"), ("teaches", "VB"), ("a", "DT"), ("basic", "JJ"), ("class", "NN")],
    [("the", "DT"), ("student", "NN"), ("studies", "VB"), ("in", "IN"), ("the", "DT"), ("lab", "NN")],
    [("the", "DT"), ("developer", "NN"), ("works", "VB"), ("in", "IN"), ("the", "DT"), ("office", "NN")],
    [("a", "DT"), ("student", "NN"), ("writes", "VB"), ("code", "NN"), ("on", "IN"), ("a", "DT"), ("laptop", "NN")],
    [("the", "DT"), ("engineer", "NN"), ("tests", "VB"), ("the", "DT"), ("system", "NN"), ("in", "IN"), ("the", "DT"), ("lab", "NN")],
    [("the", "DT"), ("teacher", "NN"), ("writes", "VB"), ("notes", "NN"), ("for", "IN"), ("the", "DT"), ("class", "NN")],
    [("a", "DT"), ("developer", "NN"), ("submits", "VB"), ("a", "DT"), ("report", "NN"), ("for", "IN"), ("the", "DT"), ("project", "NN")],
    [("the", "DT"), ("student", "NN"), ("reads", "VB"), ("a", "DT"), ("useful", "JJ"), ("book", "NN")],
    [("the", "DT"), ("professor", "NN"), ("builds", "VB"), ("an", "DT"), ("advanced", "JJ"), ("model", "NN")],
    [("a", "DT"), ("student", "NN"), ("develops", "VB"), ("software", "NN")],
    [("the", "DT"), ("developer", "NN"), ("develops", "VB"), ("software", "NN"), ("for", "IN"), ("a", "DT"), ("client", "NN")],
]

In [6]:
print(f"Training sentences: {len(TRAINING_DATA)}")


# ---------------------------------------------------------------------------
# 2-3. Extract words and labels
# ---------------------------------------------------------------------------

Training sentences: 32


In [7]:
def extract_words_and_labels(data):
    return [[w for w, t in s] for s in data], [[t for w, t in s] for s in data]


# ---------------------------------------------------------------------------

## 4. Transition frequencies (tag → tag counts)

In [8]:
# ---------------------------------------------------------------------------
def build_transition_counts(label_sequences):
    counts = defaultdict(Counter)
    for labels in label_sequences:
        prev = START
        for tag in labels:
            counts[prev][tag] += 1
            prev = tag
        counts[prev][END] += 1
    return counts


# ---------------------------------------------------------------------------

## 5. Emission frequencies (tag → word counts)

In [9]:
# ---------------------------------------------------------------------------
def build_emission_counts(data):
    counts = defaultdict(Counter)
    for sentence in data:
        for word, tag in sentence:
            counts[tag][word.lower()] += 1
    return counts


# ---------------------------------------------------------------------------

## 6. Basic probabilities + 7. Viterbi prediction
`HMMTagger` wraps the counts above into Laplace-smoothed transition and emission probabilities, then uses the classic **Viterbi algorithm** to find the single most likely tag sequence for a new sentence (rather than a greedy left-to-right guess, which can lock in an early mistake).

In [10]:
# ---------------------------------------------------------------------------
class HMMTagger:
    def __init__(self, training_data):
        self.words, self.labels = extract_words_and_labels(training_data)
        self.tagset = sorted({t for seq in self.labels for t in seq})
        self.transition_counts = build_transition_counts(self.labels)
        self.emission_counts = build_emission_counts(training_data)
        self.vocab = {w.lower() for seq in self.words for w in seq}

    def transition_prob(self, prev_tag, tag):
        counts = self.transition_counts[prev_tag]
        total = sum(counts.values())
        # Laplace smoothing over (tagset + END)
        return (counts[tag] + 1) / (total + len(self.tagset) + 1)

    def emission_prob(self, tag, word):
        counts = self.emission_counts[tag]
        total = sum(counts.values())
        word = word.lower()
        # Laplace smoothing over (vocab + 1 unseen bucket)
        return (counts[word] + 1) / (total + len(self.vocab) + 1)

    # -----------------------------------------------------------------
    # 7. Viterbi decoding: predict the most likely tag sequence
    # -----------------------------------------------------------------
    def predict(self, sentence_words):
        n = len(sentence_words)
        viterbi = [{} for _ in range(n)]
        backpointer = [{} for _ in range(n)]

        for tag in self.tagset:
            viterbi[0][tag] = (
                self.transition_prob(START, tag)
                * self.emission_prob(tag, sentence_words[0])
            )
            backpointer[0][tag] = None

        for i in range(1, n):
            word = sentence_words[i]
            for tag in self.tagset:
                best_prob, best_prev = max(
                    (
                        viterbi[i - 1][prev_tag]
                        * self.transition_prob(prev_tag, tag)
                        * self.emission_prob(tag, word),
                        prev_tag,
                    )
                    for prev_tag in self.tagset
                )
                viterbi[i][tag] = best_prob
                backpointer[i][tag] = best_prev

        last_tag = max(viterbi[-1], key=viterbi[-1].get)
        sequence = [last_tag]
        for i in range(n - 1, 0, -1):
            last_tag = backpointer[i][last_tag]
            sequence.insert(0, last_tag)
        return sequence


# ---------------------------------------------------------------------------

## Held-out evaluation data
Six sentences with manually assigned gold tags. None of these exact sentences were seen during training, so this measures generalisation, not memorisation.

In [11]:
# Evaluation dataset: sentences with manually assigned gold labels,
# none seen verbatim during training.
# ---------------------------------------------------------------------------
TEST_DATA = [
    [("the", "DT"), ("student", "NN"), ("develops", "VB"), ("software", "NN")],
    [("a", "DT"), ("teacher", "NN"), ("reads", "VB"), ("a", "DT"), ("new", "JJ"), ("book", "NN")],
    [("the", "DT"), ("engineer", "NN"), ("tests", "VB"), ("a", "DT"), ("small", "JJ"), ("database", "NN")],
    [("a", "DT"), ("student", "NN"), ("writes", "VB"), ("a", "DT"), ("report", "NN"), ("for", "IN"), ("the", "DT"), ("class", "NN")],
    [("the", "DT"), ("professor", "NN"), ("presents", "VB"), ("the", "DT"), ("results", "NN")],
    [("a", "DT"), ("developer", "NN"), ("designs", "VB"), ("an", "DT"), ("useful", "JJ"), ("application", "NN")],
]

In [12]:
def evaluate(tagger, test_data):
    total, correct = 0, 0
    all_rows = []
    errors = []
    for sentence in test_data:
        words = [w for w, t in sentence]
        gold = [t for w, t in sentence]
        predicted = tagger.predict(words)

        for word, gold_tag, pred_tag in zip(words, gold, predicted):
            total += 1
            is_correct = gold_tag == pred_tag
            correct += int(is_correct)
            all_rows.append((word, gold_tag, pred_tag, is_correct))
            if not is_correct:
                errors.append((word, gold_tag, pred_tag))

    accuracy = correct / total * 100
    return accuracy, all_rows, errors

## Run: train, predict, evaluate

In [13]:
tagger = HMMTagger(TRAINING_DATA)
print("Tagset:", tagger.tagset)
print("Vocabulary size:", len(tagger.vocab))

print("\n--- Sample prediction ---")
sample = ["the", "student", "develops", "software"]
prediction = tagger.predict(sample)
print("Sentence  :", " ".join(sample))
print("Predicted :", " ".join(prediction))

print("\n--- Evaluation on held-out test sentences ---")
accuracy, rows, errors = evaluate(tagger, TEST_DATA)

print(f"{'Word':<15}{'Gold':<8}{'Predicted':<10}{'Correct'}")
for word, gold, pred, ok in rows:
    print(f"{word:<15}{gold:<8}{pred:<10}{'yes' if ok else 'NO'}")

print(f"\nAccuracy: {accuracy:.1f}% ({sum(r[3] for r in rows)}/{len(rows)} tags correct)")

print("\n--- Errors ---")
if errors:
    for word, gold, pred in errors:
        print(f"  '{word}': predicted {pred}, expected {gold}")
else:
    print("  No errors on this test set.")


Tagset: ['DT', 'IN', 'JJ', 'NN', 'VB']
Vocabulary size: 59

--- Sample prediction ---
Sentence  : the student develops software
Predicted : DT NN VB DT

--- Evaluation on held-out test sentences ---
Word           Gold    Predicted Correct
the            DT      DT        yes
student        NN      NN        yes
develops       VB      VB        yes
software       NN      DT        NO
a              DT      DT        yes
teacher        NN      NN        yes
reads          VB      VB        yes
a              DT      DT        yes
new            JJ      JJ        yes
book           NN      NN        yes
the            DT      DT        yes
engineer       NN      NN        yes
tests          VB      VB        yes
a              DT      DT        yes
small          JJ      NN        NO
database       NN      VB        NO
a              DT      DT        yes
student        NN      NN        yes
writes         VB      VB        yes
a              DT      DT        yes
report         NN      

## 9. Comparison with manual labels & accuracy
The table above already lists, for every word in the six test sentences: the manually assigned gold tag, the tagger's predicted tag, and whether they match. Overall accuracy is printed at the end of the run above.

## 10–11. Error analysis & discussion
All three errors on the held-out set follow the same pattern: a **bare noun used as a direct object right after a verb**, with no determiner in front of it (`develops **software**`, `tests a small **database**`). Two things push the tagger toward the wrong tag here:

1. **Training-data imbalance in the VB → ? transition.** Most training sentences follow the pattern `VB DT NN` ("writes **a** report", "builds **a** system"); only a couple use a bare `VB NN` object ("develops software", "teaches students"). So `P(DT | VB)` ends up noticeably higher than `P(NN | VB)`, and for a word like *software* the emission probability alone isn't strong enough to overturn that.
2. **Sparse emissions for adjectives.** *small* was seen only once in training (as JJ), so its emission probability is weak and gets outweighed by the (smoothed) transition preference for NN in that slot; the resulting `JJ→NN` mistake then feeds a second error one step later (*database* gets tagged VB, since the model now "thinks" it just placed a noun, not an adjective).

This is a textbook HMM error mode: **a wrong tag early in a sentence propagates**, because Viterbi commits to the single globally best path, not the best path for each word independently.

## 12. Limitations
- **Small training set** (32 sentences, 5 tags) — real POS tagging uses tens of thousands of tagged sentences.
- **Unseen words** are handled only by Laplace smoothing over the whole vocabulary, which is a weak signal; a real system would use word-shape features (capitalisation, suffixes like *-s*/*-ed*) as a backoff.
- **Ambiguous words** (a word that is genuinely more than one part of speech, e.g. *books* as a noun vs. a verb) aren't represented at all in this dataset, so the tagger has never had to resolve that kind of ambiguity.
- **Limited context**: this is a first-order HMM — each tag depends only on the *immediately preceding* tag, so it cannot look further back to resolve cases like the *small database* error above.